In [1]:
import itertools
import math

import torch
from icecream import ic

In [2]:
device = torch.device("cuda")

In [3]:
positions = torch.rand(100, 100_000, 2).to(device)
bins = (500, 500)
extent_in = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
charges_in = torch.ones_like(positions[..., 0]).to(device)

In [4]:
if extent_in is None:
    extent = torch.stack([positions.amin(dim=-2), positions.amax(dim=-2)], dim=-1)
else:
    extent = extent_in
if charges_in is None:
    charges = torch.ones_like(positions[..., 0])
else:
    charges = charges_in

num_hist_dims = positions.shape[-1]
histogram_shape = [bins] * num_hist_dims if isinstance(bins, int) else bins
assert len(histogram_shape) == num_hist_dims, (
    "Number of histogram dimensions defined by bins must match number of position ",
    "tensors",
)

# Set weights to zero for particles outside grid bounds
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)

# Normalise particle coordinates to normalised bin space
bin_space_upper_bounds = torch.tensor(histogram_shape, device=positions.device) - 1
bin_widths = (extent[..., 1] - extent[..., 0]) / bin_space_upper_bounds
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

# Generate all corner combinations and their weights
corner_offsets = positions_in_bin_space_int_component.new_tensor(
    list(itertools.product([0, 1], repeat=num_hist_dims))
)  # Shape (num_corners, num_hist_dims)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    bin_space_upper_bounds.new_zeros(()), bin_space_upper_bounds - 1
)
corner_weight_factors = torch.where(
    corner_offsets == 0,
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
    positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_mask = (
    (0 <= corner_positions_in_bin_space)
    & (corner_positions_in_bin_space < bin_space_upper_bounds)
).all(dim=-1)
corner_weights = corner_weight_factors.prod(dim=-1) * corner_mask
corner_charges = corner_weights * charges.unsqueeze(-1)

vector_shape = positions.shape[:-2]
num_histogram_bins = math.prod(histogram_shape)

# Convert multi-dimensional corner positions to flat bin space indices
strides = clamped_corner_positions_in_bin_space.new_tensor(
    [math.prod(histogram_shape[i + 1 :]) for i in range(num_hist_dims)]
)
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space * strides
).sum(dim=-1)

flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

charge_grid = flat_charge_grid.reshape(*vector_shape, *histogram_shape)

final_result_to_return = charge_grid

getattr(torch, device.type).synchronize()

In [5]:
%%timeit
strides = clamped_corner_positions_in_bin_space.new_tensor(
    [math.prod(histogram_shape[i + 1 :]) for i in range(num_hist_dims)]
)

getattr(torch, device.type).synchronize()

28.5 μs ± 220 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space * strides
).sum(dim=-1)

getattr(torch, device.type).synchronize()

1.57 ms ± 171 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space * strides
)

getattr(torch, device.type).synchronize()

797 μs ± 112 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
%%timeit
corner_positions_in_flat_bin_space = (
    corner_positions_in_bin_space[..., 0]
    + histogram_shape[0] * corner_positions_in_bin_space[..., 1]
)

getattr(torch, device.type).synchronize()

1.35 ms ± 83.1 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%%timeit
flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)

getattr(torch, device.type).synchronize()

69.4 μs ± 69.1 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [10]:
%%timeit
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

getattr(torch, device.type).synchronize()

583 μs ± 676 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
%%timeit
# Normalise particle coordinates to normalised bin space
bin_space_upper_bounds = torch.tensor(histogram_shape, device=positions.device)
bin_widths = (extent[..., 1] - extent[..., 0]) / bin_space_upper_bounds
positions_in_bin_space = ((positions - extent[..., 0]) / bin_widths) - 0.5
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

getattr(torch, device.type).synchronize()

865 μs ± 215 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%%timeit
bin_space_upper_bounds = torch.tensor(histogram_shape, device=positions.device)

getattr(torch, device.type).synchronize()

29.2 μs ± 67.3 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [13]:
%%timeit
bin_widths = (extent[..., 1] - extent[..., 0]) / bin_space_upper_bounds

getattr(torch, device.type).synchronize()

35.4 μs ± 233 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [14]:
%%timeit
positions_in_bin_space = ((positions - extent[..., 0]) / bin_widths) - 0.5

getattr(torch, device.type).synchronize()

340 μs ± 134 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%%timeit
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()

getattr(torch, device.type).synchronize()

274 μs ± 103 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [16]:
%%timeit
positions_in_bin_space_int_component = positions_in_bin_space.floor()

getattr(torch, device.type).synchronize()

112 μs ± 36.4 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [17]:
%%timeit
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

getattr(torch, device.type).synchronize()

221 μs ± 78.6 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [18]:
%%timeit
corner_offsets = positions_in_bin_space_int_component.new_tensor(
    list(itertools.product([0, 1], repeat=num_hist_dims))
)  # Shape (num_corners, num_hist_dims)

getattr(torch, device.type).synchronize()

31 μs ± 261 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [19]:
%%timeit
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)

getattr(torch, device.type).synchronize()

574 μs ± 158 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
positions_in_bin_space_int_component_as_float = positions_in_bin_space_int_component.float()
corner_offsets_as_float = corner_offsets.float()

In [21]:
%%timeit
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component_as_float.unsqueeze(-2) + corner_offsets_as_float
)

getattr(torch, device.type).synchronize()

381 μs ± 72.7 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [22]:
%%timeit
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    bin_space_upper_bounds.new_zeros(()), bin_space_upper_bounds - 1
)

getattr(torch, device.type).synchronize()

820 μs ± 114 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [23]:
corner_positions_in_bin_space_as_float = corner_positions_in_bin_space.float()
bin_space_upper_bounds_as_float = bin_space_upper_bounds.float()

In [24]:
%%timeit
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space_as_float.clamp(
    bin_space_upper_bounds_as_float.new_zeros(()), bin_space_upper_bounds_as_float - 1.0
)

getattr(torch, device.type).synchronize()

464 μs ± 96.6 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [25]:
%%timeit
corner_weight_factors = torch.where(
    corner_offsets == 0,
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
    positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)

getattr(torch, device.type).synchronize()

695 μs ± 235 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [26]:
%%timeit
corner_mask = (
    (0 <= corner_positions_in_bin_space)
    & (corner_positions_in_bin_space < bin_space_upper_bounds)
).all(dim=-1)

getattr(torch, device.type).synchronize()

1.84 ms ± 257 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
%%timeit
(
    (0 <= corner_positions_in_bin_space)
    & (corner_positions_in_bin_space < bin_space_upper_bounds)
)

getattr(torch, device.type).synchronize()

1.09 ms ± 198 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [28]:
%%timeit
(0 <= corner_positions_in_bin_space)

getattr(torch, device.type).synchronize()

439 μs ± 101 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [29]:
%%timeit
(0 < corner_positions_in_bin_space)

getattr(torch, device.type).synchronize()

438 μs ± 111 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [30]:
%%timeit
(corner_positions_in_bin_space < bin_space_upper_bounds)

getattr(torch, device.type).synchronize()

502 μs ± 184 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [31]:
corner_positions_in_bin_space_as_float = corner_positions_in_bin_space.float()
bin_space_upper_bounds_as_float = bin_space_upper_bounds.float()

In [32]:
%%timeit
corner_mask = (
    (0 <= corner_positions_in_bin_space_as_float)
    & (corner_positions_in_bin_space_as_float < bin_space_upper_bounds_as_float)
).all(dim=-1)

getattr(torch, device.type).synchronize()

1.51 ms ± 193 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [33]:
%%timeit
corner_weights = corner_weight_factors.prod(dim=-1) * corner_mask

getattr(torch, device.type).synchronize()

955 μs ± 277 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [34]:
%%timeit
corner_charges = corner_weights * charges.unsqueeze(-1)

getattr(torch, device.type).synchronize()

255 μs ± 142 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [35]:
%%timeit
corner_charges = corner_weight_factors.prod(dim=-1) * corner_mask * charges.unsqueeze(-1)

getattr(torch, device.type).synchronize()

1.19 ms ± 517 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [36]:
%%timeit
vector_shape = positions.shape[:-2]
num_histogram_bins = math.prod(histogram_shape)

getattr(torch, device.type).synchronize()

5.07 μs ± 8.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [37]:
%%timeit
strides = clamped_corner_positions_in_bin_space.new_tensor(
    [math.prod(histogram_shape[i + 1 :]) for i in range(num_hist_dims)]
)

getattr(torch, device.type).synchronize()

27.9 μs ± 135 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [38]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space * strides
).sum(dim=-1)

getattr(torch, device.type).synchronize()

1.57 ms ± 461 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [39]:
clamped_corner_positions_in_bin_space_as_float = clamped_corner_positions_in_bin_space.float()
strides_as_float = strides.float()

In [40]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space_as_float * strides_as_float
).sum(dim=-1)

getattr(torch, device.type).synchronize()

1.12 ms ± 314 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [41]:
clamped_corner_positions_in_bin_space_as_32 = clamped_corner_positions_in_bin_space.int()
strides_as_32 = strides.int()

In [42]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space_as_32 * strides_as_32
).sum(dim=-1)

getattr(torch, device.type).synchronize()

1.82 ms ± 313 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [43]:
clamped_corner_positions_in_bin_space_as_16 = clamped_corner_positions_in_bin_space.short()
strides_as_16 = strides.short()

In [44]:
%%timeit
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space_as_16 * strides_as_16
).sum(dim=-1)

getattr(torch, device.type).synchronize()

1.66 ms ± 741 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [41]:
%%timeit
flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)

getattr(torch, device.type).synchronize()

70.2 μs ± 17.6 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [42]:
%%timeit
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

getattr(torch, device.type).synchronize()

595 μs ± 132 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [43]:
%%timeit
charge_grid = flat_charge_grid.reshape(*vector_shape, *histogram_shape)

getattr(torch, device.type).synchronize()

7.24 μs ± 13.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [44]:
%%timeit

if extent_in is None:
    extent = torch.stack([positions.amin(dim=-2), positions.amax(dim=-2)], dim=-1)
else:
    extent = extent_in
if charges_in is None:
    charges = torch.ones_like(positions[..., 0])
else:
    charges = charges_in

num_hist_dims = positions.shape[-1]
histogram_shape = [bins] * num_hist_dims if isinstance(bins, int) else bins
assert len(histogram_shape) == num_hist_dims, (
    "Number of histogram dimensions defined by bins must match number of position ",
    "tensors",
)

# Normalise particle coordinates to normalised bin space
bin_space_upper_bounds = torch.tensor(histogram_shape, device=positions.device)
bin_widths = (extent[..., 1] - extent[..., 0]) / bin_space_upper_bounds
positions_in_bin_space = ((positions - extent[..., 0]) / bin_widths) - 0.5
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

# Generate all corner combinations and their weights
corner_offsets = positions_in_bin_space_int_component.new_tensor(
    list(itertools.product([0, 1], repeat=num_hist_dims))
)  # Shape (num_corners, num_hist_dims)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    bin_space_upper_bounds.new_zeros(()), bin_space_upper_bounds - 1
)
corner_weight_factors = torch.where(
    corner_offsets == 0,
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
    positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_mask = (
    (0 <= corner_positions_in_bin_space)
    & (corner_positions_in_bin_space < bin_space_upper_bounds)
).all(dim=-1)
corner_weights = corner_weight_factors.prod(dim=-1) * corner_mask
corner_charges = corner_weights * charges.unsqueeze(-1)

vector_shape = positions.shape[:-2]
num_histogram_bins = math.prod(histogram_shape)

# Convert multi-dimensional corner positions to flat bin space indices
strides = clamped_corner_positions_in_bin_space.new_tensor(
    [math.prod(histogram_shape[i + 1 :]) for i in range(num_hist_dims)]
)
corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space * strides
).sum(dim=-1)

flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

charge_grid = flat_charge_grid.reshape(*vector_shape, *histogram_shape)

getattr(torch, device.type).synchronize()

8.11 ms ± 1.17 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [45]:
%%timeit

if extent_in is None:
    extent = torch.stack([positions.amin(dim=-2), positions.amax(dim=-2)], dim=-1)
else:
    extent = extent_in
if charges_in is None:
    charges = torch.ones_like(positions[..., 0])
else:
    charges = charges_in

num_hist_dims = positions.shape[-1]
histogram_shape = [bins] * num_hist_dims if isinstance(bins, int) else bins
assert len(histogram_shape) == num_hist_dims, (
    "Number of histogram dimensions defined by bins must match number of position ",
    "tensors",
)

# Set weights to zero for particles outside grid bounds
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)

# Normalise particle coordinates to normalised bin space
bin_space_upper_bounds = torch.tensor(histogram_shape, device=positions.device) - 1
bin_widths = (extent[..., 1] - extent[..., 0]) / bin_space_upper_bounds
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

# Generate all corner combinations and their weights
corner_offsets = positions_in_bin_space_int_component.new_tensor(
    list(itertools.product([0, 1], repeat=num_hist_dims))
)  # Shape (num_corners, num_hist_dims)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
).clamp(bin_space_upper_bounds.new_zeros(()), bin_space_upper_bounds)
positions_in_bin_space_fractional_components.unsqueeze(-2)
corner_weight_factors = torch.where(
    corner_offsets == 0,
    positions_in_bin_space_fractional_components.unsqueeze(-2),
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

vector_shape = positions.shape[:-2]
num_histogram_bins = math.prod(histogram_shape)

corner_positions_in_flat_bin_space = (
    corner_positions_in_bin_space[..., 0]
    + histogram_shape[0] * corner_positions_in_bin_space[..., 1]
)

flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

charge_grid = flat_charge_grid.reshape(*vector_shape, *histogram_shape)

getattr(torch, device.type).synchronize()

6.16 ms ± 1.02 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [46]:
# del extent, positions, charges

In [47]:
positions = (torch.rand(100, 100_000).to(device), torch.rand(100, 100_000).to(device))
bins = (
    torch.linspace(0.0, 3.0, 500).to(device),
    torch.linspace(0.0, 4.0, 500).to(device),
)
weights_in = torch.ones_like(positions[0]).to(device)

In [48]:
%%timeit

# Validate inputs
if not positions:
    raise ValueError("positions must contain at least one dimension")
if len(positions) != len(bins):
    raise ValueError("positions and bins must have the same length")

num_dims = len(positions)
first_pos = positions[0]
fn_device = first_pos.device
fn_dtype = first_pos.dtype

# Validate all position tensors have the same shape
for i, pos in enumerate(positions[1:], 1):
    if pos.shape != first_pos.shape:
        raise ValueError(
            f"All position tensors must have the same shape. positions[0] has shape"
            f" {first_pos.shape}, positions[{i}] has shape {pos.shape}."
        )
    if pos.device != fn_device:
        raise ValueError("All tensors must be on the same device")
    if pos.dtype != fn_dtype:
        raise ValueError("All tensors must have the same dtype")

# Grid dimensions and spacing validation
grid_sizes = []
spacings = []

for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)

if weights_in is None:
    weights = torch.ones_like(first_pos)
else:
    weights = weights_in
    
# Set weights to zero for particles outside grid bounds
for pos, bin_array in zip(positions, bins):
    outside_mask = (pos < bin_array[0]) | (pos >= bin_array[-1])
    weights = weights * (~outside_mask).float()

# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []

for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

# Generate all corner combinations and their weights
num_corners = 2**num_dims
corner_indices = []
corner_weights = []

for corner in range(num_corners):
    # Determine which corners to use based on binary representation
    corner_offsets = []
    weight_factors = []

    for dim in range(num_dims):
        if (corner >> dim) & 1:  # Use right cell in this dimension
            corner_offsets.append(1)
            weight_factors.append(fractional_parts[dim])
        else:  # Use left cell in this dimension
            corner_offsets.append(0)
            weight_factors.append(1 - fractional_parts[dim])

    # Calculate indices for this corner
    corner_idx_list = []
    for dim in range(num_dims):
        base_idx = grid_indices[dim]
        offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
        corner_idx_list.append(offset_idx)

    # Calculate weight for this corner
    corner_weight = weights
    for weight_factor in weight_factors:
        corner_weight = corner_weight * weight_factor

    corner_indices.append(corner_idx_list)
    corner_weights.append(corner_weight)

# Convert multi-dimensional indices to flat indices
def multi_to_flat_index(idx_list):
    flat_idx = idx_list[0]
    stride = 1
    for dim in range(1, num_dims):
        stride *= grid_sizes[dim - 1]
        flat_idx = flat_idx + idx_list[dim] * stride
    return flat_idx

# Flatten batch dims and particle dim together
batch_shape = first_pos.shape[:-1]
B = int(torch.tensor(batch_shape).prod()) if batch_shape else 1
N = first_pos.shape[-1]

def flatten_tensor(t):
    return t.reshape(B, N)

# Prepare all indices and weights for batch processing
all_flat_indices = []
all_weights = []

for corner_idx_list, corner_weight in zip(corner_indices, corner_weights):
    flat_idx = multi_to_flat_index(corner_idx_list)
    all_flat_indices.append(flatten_tensor(flat_idx))
    all_weights.append(flatten_tensor(corner_weight))

# Concatenate all indices and weights
idx_all = torch.cat(all_flat_indices, dim=1)  # shape (B, num_corners * N)
vals_all = torch.cat(all_weights, dim=1)  # shape (B, num_corners * N)

# Output buffer
total_grid_size = int(torch.tensor(grid_sizes).prod())
charge = torch.zeros((B, total_grid_size), dtype=fn_dtype, device=fn_device)

# Vectorized batched index_add_
for b in range(B):
    charge[b].index_add_(0, idx_all[b], vals_all[b])

# Compute inverse cell volume
# cell_volume = 1.0
# for spacing in spacings:
#     cell_volume *= spacing
# inv_cell_volume = 1.0 / cell_volume
# charge = charge * inv_cell_volume

# Reshape back to original batch dimensions + grid dimensions
out_shape = (*batch_shape, *grid_sizes[::-1])
charge = charge.reshape(out_shape)  # Grid dimensions are reversed by the reshape

batch_ndim = len(batch_shape)
spatial_axes = list(range(batch_ndim, batch_ndim + num_dims))

# Permute to put spatial axes in the correct order
final_result_to_return = charge.permute(*range(batch_ndim), *reversed(spatial_axes))

getattr(torch, device.type).synchronize()

7.24 ms ± 1.39 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [49]:
%%timeit

# Validate inputs
if not positions:
    raise ValueError("positions must contain at least one dimension")
if len(positions) != len(bins):
    raise ValueError("positions and bins must have the same length")

num_dims = len(positions)
first_pos = positions[0]
fn_device = first_pos.device
fn_dtype = first_pos.dtype

# Validate all position tensors have the same shape
for i, pos in enumerate(positions[1:], 1):
    if pos.shape != first_pos.shape:
        raise ValueError(
            f"All position tensors must have the same shape. positions[0] has shape"
            f" {first_pos.shape}, positions[{i}] has shape {pos.shape}."
        )
    if pos.device != fn_device:
        raise ValueError("All tensors must be on the same device")
    if pos.dtype != fn_dtype:
        raise ValueError("All tensors must have the same dtype")

# Grid dimensions and spacing validation
grid_sizes = []
spacings = []

for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)

if weights_in is None:
    weights = torch.ones_like(first_pos)
else:
    weights = weights_in
    
# Set weights to zero for particles outside grid bounds
for pos, bin_array in zip(positions, bins):
    outside_mask = (pos < bin_array[0]) | (pos >= bin_array[-1])
    weights = weights * (~outside_mask).float()

# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []

for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

# Generate all corner combinations and their weights
num_corners = 2**num_dims
corner_indices = []
corner_weights = []

for corner in range(num_corners):
    # Determine which corners to use based on binary representation
    corner_offsets = []
    weight_factors = []

    for dim in range(num_dims):
        if (corner >> dim) & 1:  # Use right cell in this dimension
            corner_offsets.append(1)
            weight_factors.append(fractional_parts[dim])
        else:  # Use left cell in this dimension
            corner_offsets.append(0)
            weight_factors.append(1 - fractional_parts[dim])

    # Calculate indices for this corner
    corner_idx_list = []
    for dim in range(num_dims):
        base_idx = grid_indices[dim]
        offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
        corner_idx_list.append(offset_idx)

    # Calculate weight for this corner
    corner_weight = weights
    for weight_factor in weight_factors:
        corner_weight = corner_weight * weight_factor

    corner_indices.append(corner_idx_list)
    corner_weights.append(corner_weight)

# Convert multi-dimensional indices to flat indices
def multi_to_flat_index(idx_list):
    flat_idx = idx_list[0]
    stride = 1
    for dim in range(1, num_dims):
        stride *= grid_sizes[dim - 1]
        flat_idx = flat_idx + idx_list[dim] * stride
    return flat_idx

# Flatten batch dims and particle dim together
batch_shape = first_pos.shape[:-1]
B = int(torch.tensor(batch_shape).prod()) if batch_shape else 1
N = first_pos.shape[-1]

def flatten_tensor(t):
    return t.reshape(B, N)

# Prepare all indices and weights for batch processing
all_flat_indices = []
all_weights = []

for corner_idx_list, corner_weight in zip(corner_indices, corner_weights):
    flat_idx = multi_to_flat_index(corner_idx_list)
    all_flat_indices.append(flatten_tensor(flat_idx))
    all_weights.append(flatten_tensor(corner_weight))

# Concatenate all indices and weights
idx_all = torch.cat(all_flat_indices, dim=1)  # shape (B, num_corners * N)
vals_all = torch.cat(all_weights, dim=1)  # shape (B, num_corners * N)

# Output buffer
total_grid_size = int(torch.tensor(grid_sizes).prod())
charge = torch.zeros((B, total_grid_size), dtype=fn_dtype, device=fn_device)

# Vectorized batched index_add_
for b in range(B):
    charge[b].index_add_(0, idx_all[b], vals_all[b])

# Compute inverse cell volume
# cell_volume = 1.0
# for spacing in spacings:
#     cell_volume *= spacing
# inv_cell_volume = 1.0 / cell_volume
# charge = charge * inv_cell_volume

# Reshape back to original batch dimensions + grid dimensions
out_shape = (*batch_shape, *grid_sizes[::-1])
charge = charge.reshape(out_shape)  # Grid dimensions are reversed by the reshape

batch_ndim = len(batch_shape)
spatial_axes = list(range(batch_ndim, batch_ndim + num_dims))

# Permute to put spatial axes in the correct order
final_result_to_return = charge.permute(*range(batch_ndim), *reversed(spatial_axes))

getattr(torch, device.type).synchronize()

7.24 ms ± 1.73 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [50]:
# del positions, bins, weights

In [51]:
positions = torch.rand(100, 100_000, 2).to(device)
bins = (500, 500)
extent_in = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
charges_in = torch.ones_like(positions[..., 0]).to(device)

In [52]:
%%timeit

if extent_in is None:
    extent = torch.stack([positions.amin(dim=-2), positions.amax(dim=-2)], dim=-1)
else:
    extent = extent_in
if charges_in is None:
    charges = torch.ones_like(positions[..., 0])
else:
    charges = charges_in

num_hist_dims = positions.shape[-1]
histogram_shape = [bins] * num_hist_dims if isinstance(bins, int) else bins
assert len(histogram_shape) == num_hist_dims, (
    "Number of histogram dimensions defined by bins must match number of position ",
    "tensors",
)

bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)

# Set weights to zero for particles outside grid bounds
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)

# Normalise particle coordinates to normalised bin space
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

# Generate all corner combinations and their weights
num_corners = 2**num_hist_dims
# TODO speed: torch.tensor(list(itertools.product([0, 1], repeat=D)), device=device)
corner_offsets = (
    torch.arange(num_corners).unsqueeze(-1).to(positions.device)
    // (2 ** torch.arange(num_hist_dims).to(positions.device))
    % 2
)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    corner_positions_in_bin_space.new_zeros(()),
    corner_positions_in_bin_space.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = positions_in_bin_space_fractional_components.unsqueeze(
    -2
).where(
    corner_offsets == 1,
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

vector_shape = positions.shape[:-2]
num_histogram_bins = math.prod(histogram_shape)

corner_positions_in_flat_bin_space = (
    clamped_corner_positions_in_bin_space[..., 0]
    + histogram_shape[0] * clamped_corner_positions_in_bin_space[..., 1]
)

flat_charge_grid = corner_charges.new_zeros(*vector_shape, num_histogram_bins)
flat_charge_grid.scatter_add_(
    dim=-1,
    index=corner_positions_in_flat_bin_space.flatten(start_dim=-2),
    src=corner_charges.flatten(start_dim=-2),
)

charge_grid = flat_charge_grid.reshape(*vector_shape, *histogram_shape)

final_result_to_return = charge_grid

getattr(torch, device.type).synchronize()

6.26 ms ± 1.03 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [53]:
# del extent, positions, charges